# v0.3.0 — Operational Model Governance

**Projeto:** Renewable Energy MLOps — Wind Power Forecasting  
**Versão:** v0.3.0  
**Branch:** `ops/model-governance`  
**Status:** etapa de governança operacional concluída  
**Champion anterior:** v10 — `d466ccb7fa294851a86a73b05d4e5735`  
**Challenger promovido:** v17 — `1d13a61244c54f06aa70f43a9993ea37`  
**OOT de decisão:** setembro/2026

Este notebook documenta a evolução do projeto de uma promoção baseada em métricas
históricas para um **Quality Gate com comparação Champion × Challenger sob o mesmo
conjunto Out-of-Time (same-OOT)**.

A v0.3 do projeto é composta por três frentes operacionais:

```text
v0.3
├── Operational Governance
├── Longitudinal Monitoring
└── Client / E2E Validation
```

Este notebook trata especificamente de **Operational Governance**. Caso Monitoring
longitudinal ou validação Client/E2E cresçam o suficiente para exigir narrativa
própria, serão mantidos em notebooks separados da mesma versão. O notebook principal
permanece como síntese curada da evolução completa do sistema.

## 1. Objetivo

O objetivo desta etapa é tornar a promoção de modelos uma decisão operacional
**comparável, reproduzível e auditável**.

O problema identificado no lifecycle anterior era conceitual: o Challenger podia ser
avaliado no OOT atual enquanto o Champion era representado por uma métrica histórica
registrada em outra execução. Ainda que os dois números fossem válidos em seus
respectivos experimentos, eles não constituíam necessariamente uma comparação justa
para promoção automática.

A política da v0.3 passa a exigir:

- Champion e Challenger avaliados sobre o **mesmo OOT congelado**;
- mesma implementação de cálculo de MAE e nMAE;
- identidade explícita das versões e Runs;
- decisão automática associada a uma versão de política;
- metadados auditáveis persistidos no MLflow;
- promoção somente após o Quality Gate;
- hot reload do serviço somente depois da promoção.

## 2. Contexto experimental anterior

A v0.2.0 produziu duas linhas principais de evidência experimental:

1. **Simplificação do ensemble** — comparação controlada entre arquiteturas;
2. **Comparação de janelas de treinamento** — expanding, rolling 24 meses e rolling
   12 meses sob protocolo temporal comum.

No benchmark de janelas, a **expanding window** apresentou a melhor generalização OOT
entre os três tratamentos para setembro/2026. O modelo correspondente foi registrado
como **v17**.

Esse benchmark foi deliberadamente isolado do lifecycle operacional: nenhuma das três
janelas alterou automaticamente o alias `@champion`. Assim, a escolha experimental da
v17 como principal Challenger permaneceu separada da decisão de promoção.

## 3. Refatoração da governança

A principal mudança da v0.3 não foi adicionar uma nova arquitetura de Machine
Learning, mas corrigir **como modelos são comparados antes de uma promoção**.

### Antes

```text
Champion
└── métrica histórica da Run

Challenger
└── métrica da execução atual

        ↓

comparação potencialmente realizada
sob contextos OOT diferentes
```

### Depois

```text
Champion ─────┐
              ├── mesmo OOT congelado
Challenger ───┘
        ↓
shared evaluate_model_on_oot()
        ↓
automatic Quality Gate
        ↓
PROMOTE / REJECT
        ↓
auditable MLflow metadata
```

As melhorias implementadas nesta etapa incluem:

- extração de `evaluate_model_on_oot()` para um componente compartilhado de avaliação;
- alinhamento das features segundo o contrato esperado pelo modelo carregado;
- reutilização do mesmo contrato de avaliação pelo Monitoring;
- comparação same-OOT no Quality Gate;
- preservação de métricas históricas como evidência e baseline, sem utilizá-las como
  critério direto de promoção quando o OOT é diferente;
- `governance_policy_version = same_oot_v1`;
- registro do Champion anterior, Challenger, métricas same-OOT, deltas, OOT, decisão
  e motivo no MLflow;
- testes automatizados protegendo promoção, rejeição e regressões do comportamento
  same-OOT.

O escopo foi mantido deliberadamente simples. Não foi adicionada uma plataforma de
aprovação humana ou manual override, pois isso não resolve um problema concreto desta
fase do projeto.

## 4. Comparação definitiva — Champion v10 vs Challenger v17

A comparação operacional definitiva utiliza um único OOT congelado:

```text
OOT único
2026-09-01 03:00 → 2026-10-01 02:00
720 observações

   v10 (Champion) ─────┐
                       ├── evaluate_model_on_oot()
   v17 (Challenger) ───┘

Comparação controlada:
- mesmos dados
- mesmo target
- mesma capacidade
- mesma implementação de métricas
```

O intervalo temporal conceitual do mês é:

```text
[2026-09-01 03:00:00, 2026-10-01 03:00:00)
```

Como os dados são horários, a última observação presente é
`2026-10-01 02:00:00`.

### 4.1. Configuração reproduzível

A célula abaixo fixa explicitamente as duas versões e o snapshot OOT. Ela não utiliza
o alias `@champion` para carregar os modelos avaliados, evitando que uma mudança futura
do alias altere retrospectivamente o experimento.

In [1]:
import os

import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.tracking import MlflowClient

from energy_mlops.config import settings
from energy_mlops.data.feature_utils import select_model_features
from energy_mlops.models.evaluation import evaluate_model_on_oot

MODEL_NAME = "ensemble_lgb_xgb_rf_bahia"

CHAMPION_VERSION = "10"
CHAMPION_RUN_ID = "d466ccb7fa294851a86a73b05d4e5735"

CHALLENGER_VERSION = "17"
CHALLENGER_RUN_ID = "1d13a61244c54f06aa70f43a9993ea37"

OOT_DATASET = "oot_test_wind_energy_2026_09.parquet"
OOT_PATH = (
    f"s3://{settings.RUSTFS_BUCKET}/gold/"
    f"{OOT_DATASET}"
)

os.environ["AWS_ACCESS_KEY_ID"] = settings.RUSTFS_ROOT_USER
os.environ["AWS_SECRET_ACCESS_KEY"] = settings.RUSTFS_ROOT_PASSWORD
os.environ["MLFLOW_S3_ENDPOINT_URL"] = settings.RUSTFS_ENDPOINT
os.environ["AWS_ENDPOINT_URL"] = settings.RUSTFS_ENDPOINT

mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)

client = MlflowClient(
    tracking_uri=settings.MLFLOW_TRACKING_URI
)

/home/wanderson/.cache/pypoetry/virtualenvs/energy-mlops-G3NdcX_6-py3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 4.2. Identidade dos modelos e validação do OOT

As versões são verificadas contra os `run_id` congelados. O OOT também é validado em
número de linhas e limites temporais antes da avaliação.

In [2]:
champion_info = client.get_model_version(
    MODEL_NAME,
    CHAMPION_VERSION,
)

challenger_info = client.get_model_version(
    MODEL_NAME,
    CHALLENGER_VERSION,
)

assert champion_info.run_id == CHAMPION_RUN_ID
assert challenger_info.run_id == CHALLENGER_RUN_ID

df_oot = pd.read_parquet(
    OOT_PATH,
    storage_options=settings.storage_options,
)

dates = pd.to_datetime(
    df_oot["date"],
    errors="raise",
)

assert len(df_oot) == 720
assert dates.min() == pd.Timestamp("2026-09-01 03:00:00")
assert dates.max() == pd.Timestamp("2026-10-01 02:00:00")

X_oot = select_model_features(df_oot)

print("OOT:", OOT_DATASET)
print("rows:", len(df_oot))
print("start:", dates.min())
print("end:", dates.max())
print("features:", X_oot.columns.tolist())

OOT: oot_test_wind_energy_2026_09.parquet
rows: 720
start: 2026-09-01 03:00:00
end: 2026-10-01 02:00:00
features: ['temperature_2m', 'wind_speed_100m', 'wind_direction_100m', 'wind_temp_ratio', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'wind_speed_roll_mean_3h']


### 4.3. Avaliação same-OOT

Os dois modelos são carregados por **versão explícita** e avaliados pela mesma função.

In [3]:
champion_model = mlflow.sklearn.load_model(
    f"models:/{MODEL_NAME}/{CHAMPION_VERSION}"
)

challenger_model = mlflow.sklearn.load_model(
    f"models:/{MODEL_NAME}/{CHALLENGER_VERSION}"
)

champion_eval = evaluate_model_on_oot(
    champion_model,
    X_oot,
    df_oot,
)

challenger_eval = evaluate_model_on_oot(
    challenger_model,
    X_oot,
    df_oot,
)

mae_delta_mw = (
    challenger_eval.mae_mw
    - champion_eval.mae_mw
)

nmae_delta_pp = (
    challenger_eval.nmae_pct
    - champion_eval.nmae_pct
)

mae_improvement_pct = (
    (
        champion_eval.mae_mw
        - challenger_eval.mae_mw
    )
    / champion_eval.mae_mw
    * 100
)

### 4.4. Tabela comparativa

A saída abaixo é gerada diretamente das avaliações same-OOT e pode ser reproduzida
sempre que os artifacts originais permanecerem disponíveis.

In [4]:
comparison = pd.DataFrame(
    {
        "Model": [
            "v10 — Champion anterior",
            "v17 — Challenger",
        ],
        "MAE (MW)": [
            champion_eval.mae_mw,
            challenger_eval.mae_mw,
        ],
        "nMAE (%)": [
            champion_eval.nmae_pct,
            challenger_eval.nmae_pct,
        ],
    }
)

comparison["Δ MAE vs v10 (MW)"] = (
    comparison["MAE (MW)"]
    - champion_eval.mae_mw
)

comparison["Δ nMAE vs v10 (p.p.)"] = (
    comparison["nMAE (%)"]
    - champion_eval.nmae_pct
)

display(
    comparison.style.format(
        {
            "MAE (MW)": "{:.3f}",
            "nMAE (%)": "{:.4f}",
            "Δ MAE vs v10 (MW)": "{:+.3f}",
            "Δ nMAE vs v10 (p.p.)": "{:+.4f}",
        }
    )
)

print(
    "MAE relative improvement:",
    f"{mae_improvement_pct:.4f}%"
)

,Model,MAE (MW),nMAE (%),Δ MAE vs v10 (MW),Δ nMAE vs v10 (p.p.)
0,v10 — Champion anterior,803.372,6.8162,+0.000,+0.0000
1,v17 — Challenger,787.184,6.6788,-16.189,-0.1374


MAE relative improvement: 2.0151%


### 4.5. Resultado observado

A execução real do protocolo produziu:

| Métrica | v10 — Champion anterior | v17 — Challenger | Δ v17 − v10 |
|---|---:|---:|---:|
| MAE (MW) | 803.3723 | **787.1836** | **−16.1887 MW** |
| nMAE (%) | 6.8162 | **6.6788** | **−0.1374 p.p.** |

A redução relativa do MAE foi de aproximadamente **2.0151%**.

A métrica histórica da v10 registrada em sua Run era `829.5865 MW`. Quando a v10 foi
reavaliada no mesmo OOT da v17, seu MAE foi `803.3723 MW`. Essa diferença demonstra
por que métricas históricas de contextos distintos não devem, por si só, decidir uma
promoção automática.

A v17 reproduziu `787.1836 MW` e `6.6788%`, consistentes com o benchmark original da
expanding window, fornecendo uma verificação adicional de reprodutibilidade.

**Conclusão:** sob o OOT congelado de setembro/2026 e a política `same_oot_v1`, a v17
satisfaz o critério primário `MAE_IMPROVED_SAME_OOT`.

Este resultado não implica superioridade universal da v17; a evidência é específica
ao protocolo e ao OOT avaliados.

## 5. Promoção operacional auditável

Depois da comparação read-only, a v17 foi submetida ao lifecycle operacional real.

O Quality Gate reavaliou Champion e Challenger no mesmo OOT e registrou:

```text
quality_gate_status = PROMOTED_CHAMPION
governance_final_decision = PROMOTE
governance_decision_reason = MAE_IMPROVED_SAME_OOT
governance_policy_version = same_oot_v1
```

Metadados principais persistidos na Run da v17:

| Campo | Valor |
|---|---|
| Champion anterior | v10 |
| Challenger | v17 |
| Champion MAE same-OOT | 803.372317 MW |
| Challenger MAE same-OOT | 787.183586 MW |
| Δ MAE | −16.188731 MW |
| Champion nMAE same-OOT | 6.816154% |
| Challenger nMAE same-OOT | 6.678802% |
| Δ nMAE | −0.137352 p.p. |
| OOT | `oot_test_wind_energy_2026_09.parquet` |
| OOT rows | 720 |
| Política | `same_oot_v1` |

A mudança operacional seguiu o fluxo:

```text
v10 @champion
      ↓
same-OOT Quality Gate
      ↓
v17 MAE_IMPROVED_SAME_OOT
      ↓
auditable governance metadata
      ↓
MLflow alias @champion → v17
      ↓
hot reload
      ↓
API serving v17
```

O hot reload respondeu com sucesso e a API passou a reportar a v17 como modelo ativo.

### 5.1. Inspeção read-only do estado atual

Esta célula **não promove modelos**. Ela apenas consulta o Registry e os metadados
persistidos para auditoria.

In [5]:
champion_now = client.get_model_version_by_alias(
    MODEL_NAME,
    "champion",
)

promoted_run = client.get_run(
    CHALLENGER_RUN_ID
)

governance_keys = [
    "quality_gate_status",
    "governance_final_decision",
    "governance_decision_reason",
    "governance_policy_version",
    "governance_previous_champion_version",
    "governance_challenger_version",
    "governance_champion_mae_mw_same_oot",
    "governance_challenger_mae_mw_same_oot",
    "governance_mae_delta_mw_same_oot",
    "governance_champion_nmae_pct_same_oot",
    "governance_challenger_nmae_pct_same_oot",
    "governance_nmae_delta_pp_same_oot",
    "governance_oot_dataset",
    "governance_oot_rows",
    "governance_oot_start_utc",
    "governance_oot_end_utc",
]

governance_state = {
    key: promoted_run.data.tags.get(key)
    for key in governance_keys
}

print(
    "Current @champion:",
    f"v{champion_now.version}",
    champion_now.run_id,
)

pd.Series(
    governance_state,
    name="value",
).to_frame()

Current @champion: v17 1d13a61244c54f06aa70f43a9993ea37


,value
quality_gate_status,PROMOTED_CHAMPION
governance_final_decision,PROMOTE
governance_decision_reason,MAE_IMPROVED_SAME_OOT
governance_policy_version,same_oot_v1
governance_previous_champion_version,10
governance_challenger_version,17
governance_champion_mae_mw_same_oot,803.3723172253395
governance_challenger_mae_mw_same_oot,787.1835861175788
governance_mae_delta_mw_same_oot,-16.18873110776076
governance_champion_nmae_pct_same_oot,6.816153646397424


### 5.2. Evidência do serving

Após o hot reload, `/model-info` retornou:

```json
{
  "model_name": "ensemble_lgb_xgb_rf_bahia",
  "alias": "champion",
  "version": 17,
  "run_id": "1d13a61244c54f06aa70f43a9993ea37",
  "metrics": {
    "oot_mae_mw": 787.1835861175788,
    "oot_nmae_pct": 6.678801541769502,
    "oot_mae_fc_pct": 6.678801541769504,
    "oot_r2_score": 0.8613955426061799
  }
}
```

Isso fecha a cadeia entre decisão de governança, Model Registry e estado servido pela
API.

## 6. Conclusões da etapa de Operational Governance

A principal conclusão desta etapa não é apenas que a v17 apresentou menor MAE.

A contribuição operacional da v0.3 foi estabelecer um **protocolo de promoção**:

1. escolher um Challenger com base em evidência experimental;
2. congelar e identificar o OOT de decisão;
3. reavaliar Champion e Challenger nas mesmas condições;
4. aplicar uma política explícita de Quality Gate;
5. persistir a decisão e sua proveniência;
6. alterar o alias somente depois da aprovação;
7. atualizar o serving por hot reload;
8. verificar que a API está servindo a versão promovida.

No OOT de setembro/2026, a v17 reduziu o MAE em `16.1887 MW` e o nMAE em
`0.1374 p.p.` em relação à v10. A promoção ocorreu pelo critério
`MAE_IMPROVED_SAME_OOT`.

A v17 passa, portanto, a ser o Champion operacional do projeto para a próxima etapa da
v0.3.

## 7. Próximas frentes da v0.3

A próxima fase é **Longitudinal Monitoring**, com foco em evidência ao longo do tempo:

```text
Longitudinal Monitoring
├── data drift por mês
├── performance drift por mês
└── histórico Champion / Challenger
```

Depois será executada a validação **Client / E2E**, cobrindo a fronteira pública do
sistema e a consistência entre Registry, reload e modelo efetivamente servido.

Se essas frentes crescerem o suficiente para comprometer a clareza deste notebook,
serão documentadas em notebooks próprios da v0.3. Ao final da versão, as conclusões
serão sintetizadas no notebook principal do projeto.